## Model and answer-grounding upgrade

This study tests stronger local generators and retrieval models, explicit source-linked claims, evidence verification, abstention calibration, and sequential retrieval. It preserves the original study at commit 92a85e1. Results distinguish new held-out questions from the original comparison and from smaller controlled subsets.

In [ ]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
get_ipython().run_line_magic('matplotlib', 'inline')
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from advanced_rag.data import load
from advanced_rag.pipeline import answer_scores
from advanced_rag.upgrade.artifacts import read, preserved, sources, fingerprint
from advanced_rag.upgrade.evaluate import aggregate, per_question, cluster_interval
from advanced_rag.upgrade.grounding import gate
OUT = ROOT / 'reports/upgrade'
LOCAL = ROOT / 'data/upgrade'
metrics = read(OUT / 'metrics.json')
selection = read(OUT / 'selection.json')
config = read(ROOT / 'configs/upgrade-study.json')
frames = {split: {name: pd.DataFrame(methods).T for name, methods in datasets.items()} for split, datasets in metrics.items()}
def records(label, name, split):
    manifest = read(LOCAL / 'manifests' / f'{label}-{name}-{split}.json')
    return manifest, [read(ROOT / item['path'])['result'] for item in manifest['rows']]
plt.rcParams.update({'figure.figsize': (10, 4), 'axes.spines.top': False, 'axes.spines.right': False})
display(pd.DataFrame(config.items(), columns=['setting','value']))

## Preserve the original evidence

All original tracked files, local experiment traces, and the executed baseline notebook have preservation hashes. The baseline notebook also has a local backup. The upgrade has a separate implementation, reports, and notebook.

In [ ]:
preserved()
preservation = read(OUT / 'preservation.json')
print('Original commit:', preservation['commit'])
print('Preserved files:', len(preservation['files']))
print('Backup notebook hash:', preservation['notebook_backup_sha256'])

## Confirmed baseline weaknesses

The original code uses FLAN-T5-small, one post hoc lexical citation, frequent-term expansion, and overlap thresholds selected on 24 development questions. Its 20-case assistant review contains 9 unsupported answers. This review is not an independent human annotation set.

In [ ]:
original_metrics = read(ROOT / 'reports/metrics.json')
display(read(ROOT / 'reports/manual-support-review.json')['counts'])
display({name: read(ROOT / f'reports/{name}-selection.json')['selected_threshold'] for name in original_metrics})

## Research questions

The study asks whether stronger models improve answers under fixed retrieval, whether better evidence selection survives generation, and whether explicit citations plus NLI verification reduce unsupported accepted outputs. It also tests evidence-dependent follow-up retrieval. A smaller generator can win; negative results remain in the comparison.

In [ ]:
print('Generator candidates:', config['generator_candidates'])
print('Retrieval candidates:', [r['name'] for r in config['retrieval_candidates']])

## Hardware before model selection

A small original-model CPU workload preceded new model downloads. Resource observations describe this machine at audit time. The CUDA overlay is separate from the original package directory.

In [ ]:
hardware = read(OUT / 'hardware-initial.json')
display({k: v for k, v in hardware.items() if k != 'packages'})

## CPU feasibility workload

The eight baseline development questions measure actual loading, indexing, token counts, and answer latency. They are a feasibility probe, not a held-out benchmark.

In [ ]:
probe = read(OUT / 'baseline-feasibility.json')
print('Loading seconds:', probe['loading_seconds'])
display(pd.DataFrame(probe['workload']))

## GPU feasibility workload

Qwen3-0.6B and 1.7B were tested before the expanded study. The initial structured prompt and its shorter revision differ, so probe quality scores should not be compared as a controlled accuracy experiment. Formal study comparisons use fixed prompts.

In [ ]:
feasibility = read(OUT / 'feasibility.json')
display(pd.DataFrame({name: {'load_ms': r['load_ms'], 'gpu_peak_bytes': r['gpu_peak_bytes'], 'rss_bytes': r['rss_bytes']} for name, r in feasibility['generators'].items()}).T)

## Model licenses and immutable revisions

The candidate decisions and primary model-card links are in docs/upgrade-design.md. Model weights remain local. The lock records snapshot revisions, file hashes, and observed download sizes.

In [ ]:
models = read(ROOT / 'models.upgrade.lock.json')
display(pd.DataFrame({name: {k: r[k] for k in ['id','role','revision','license','bytes']} for name, r in models.items()}).T)

## Deferred models

Unquantized Qwen3-4B leaves insufficient GPU space for this workflow's supporting models and runtime allocations. Quantized 4B inference was not tested. BGE-base and the larger BGE reranker were assessed but not downloaded; the experiment compares BGE-small and MiniLM reranking with the original models.

In [ ]:
print('Downloaded upgrade snapshots:', list(models))
assert not any('Qwen3-4B' in r['id'] for r in models.values())

## Qwen adapter settings

Qwen uses the documented chat template with thinking disabled. Non-thinking generation uses temperature 0.7, top-p 0.8, and top-k 20. FLAN uses its sequence-to-sequence adapter and greedy decoding. This difference is part of the declared model configuration.

In [ ]:
import inspect
from advanced_rag.upgrade.models import Generator
source = inspect.getsource(Generator.prompt) + inspect.getsource(Generator.generate)
assert 'enable_thinking=False' in source
assert 'temperature=.7' in source
print(source)

## Dataset identity

The underlying HotpotQA and SQuAD releases are unchanged. HotpotQA uses supplied distractor passages, and SQuAD uses each original paragraph. Evaluation still measures bounded evidence selection, not open-domain retrieval.

In [ ]:
display(pd.DataFrame(read(ROOT / 'sources.lock.json'))[['name','license','sha256','bytes']])

## Fresh and original questions

The next 256 questions after the original 64 form the fresh evaluation sample for each dataset. Development grows to 64 questions. Original questions are evaluated separately, allowing direct before-and-after comparisons without using them for selection.

In [ ]:
datasets = {name: load(name, 64, 320) for name in ('hotpot','squad')}
for name, (dev, test, info) in datasets.items():
    assert not {r['id'] for r in test[:64]} & {r['id'] for r in test[64:]}
    assert not {r['group'] for r in dev} & {r['group'] for r in test}
    print(name, 'development', len(dev), 'original', len(test[:64]), 'fresh', len(test[64:]), 'fresh groups', len({r['group'] for r in test[64:]}))

## Sampling trace

Published ID manifests allow the sample to be recreated. No sampling rule filters by model correctness, answerability, or support score.

In [ ]:
for name in datasets:
    sampling = read(OUT / f'{name}-sampling.json')
    assert sampling['fresh_ids'] == [r['id'] for r in datasets[name][1][64:]]
    print(name, 'sampling identity verified', len(sampling['fresh_ids']))

## Answerability and question types

SQuAD unanswerability comes from the original annotations. HotpotQA bridge and comparison types are retained. The selected sample is not balanced by outcome.

In [ ]:
display(pd.DataFrame([{'dataset': name, 'kind': r['kind']} for name, (_, test, _) in datasets.items() for r in test[64:]]).value_counts().rename('questions').to_frame())

## Baseline reproduction

The unchanged CPU baseline is rerun on original questions and extended to the fresh sample. Exact answer agreement is checked separately from latency because background load can change timing.

In [ ]:
baseline = read(OUT / 'baseline-reproduction.json')
display({name: r['answer_exact_reproduction'] for name, r in baseline.items()})

## Frozen experiment identity

Predictions are cached by question content, configuration, model locks, source hashes, and environment identity. A changed source or setting produces a new cache key. Resumed stages retain the original inference timings.

In [ ]:
identity = read(OUT / 'experiment-identity.json')
assert identity['source_sha256'] == sources()
print('Study fingerprint:', identity['fingerprint'])
display({k: v for k, v in identity.items() if k not in ('source_sha256','config')})

## Chunking and source mapping

The upgrade retains 100-word windows and 20-word overlap. Retrieval includes source titles and maps window IDs back to original documents. This representation change is distinct from the encoder comparison and is disclosed when comparing with the original baseline.

In [ ]:
from advanced_rag.pipeline import chunk
lengths = []
for name, (_, test, _) in datasets.items():
    for row in test[64:]:
        windows, mapping = chunk(row['docs'])
        assert set(mapping.values()) == set(row['docs'])
        lengths.extend({'dataset': name, 'words': len(text.split())} for text in windows.values())
display(pd.DataFrame(lengths).groupby('dataset').words.describe())

## Plain-answer generator comparison

All four generators receive the same retrieval configuration and context policy. FLAN's prompt family and Qwen's chat adapter remain model-specific. Citation-capable selection is a separate structured-output comparison.

In [ ]:
for name, frame in frames['development'].items():
    display(frame.loc[[m for m in frame.index if m.startswith('generator_plain_')], ['n','em','f1','coverage','p50_ms','output_tokens']])

## Controlled generator results

The held-out comparison uses the same 64 fresh questions per dataset, keeping retrieval fixed. Plain answers do not receive post hoc citations in this stage.

In [ ]:
for name, frame in frames['controlled'].items():
    display(frame.loc[[m for m in frame.index if m.startswith('generator_plain_')], ['n','em','f1','answerable_f1','unanswerable_recall','p50_ms']])

## Generator quality plot

Point estimates describe the controlled subset. A higher-capacity model is not assumed to win, and a lower coverage can raise SQuAD performance through abstention.

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,4))
for ax, (name, frame) in zip(axes, frames['controlled'].items()):
    frame.loc[[m for m in frame.index if m.startswith('generator_plain_')], ['em','f1']].plot.bar(ax=ax); ax.set_title(name); ax.set_ylim(0,1)
plt.tight_layout(); plt.show()

## Structured generator selection

The two Qwen candidates are compared on their ability to produce an answer with explicit source-linked claims. FLAN's unsuccessful JSON feasibility outputs remain recorded. Selecting among citation-capable generators does not imply that the selected generator beats FLAN on plain-answer accuracy.

In [ ]:
display(pd.DataFrame(selection['structured_generators'])[['generator','f1','coverage','valid','p50_ms']])
print('Selected citation-capable generator:', selection['selected_generator'])

## Citation parsing and rejection

Every cited ID must occur in the actual prompt evidence. Missing claims, malformed JSON, fabricated source IDs, and an answer absent from its claims cause rejection. These rules check structure and consistency, not truth.

In [ ]:
from advanced_rag.upgrade.grounding import parse
e = {'S1': {'text':'The capital of France is Paris.', 'document_id':'France'}}
assert parse('{"answer":"Paris","claims":[{"text":"Paris is the capital of France.","sources":["S99"]}]}', e)['reason'] == 'fabricated_citation'
print('Fabricated source rejected')

## NLI support verification

The verifier checks each complete claim against its combined cited windows and each individual citation. The minimum combined-claim score controls support gating. Model-based entailment can be wrong and must remain separate from source-label agreement.

In [ ]:
from advanced_rag.upgrade.grounding import verify
print(inspect.getsource(verify))

## Retrieval development selection

BM25, dense retrieval, hybrid fusion, BGE embeddings, and two rerankers are compared with the selected generator fixed. Only development answer F1 chooses the retrieval setting.

In [ ]:
display(pd.DataFrame([{**r['configuration'], 'f1': r['f1'], 'evidence_recall': r['evidence_recall'], 'p50_ms':r['p50_ms']} for r in selection['retrieval']]))

## Controlled retrieval comparison

Each method uses the same first 64 fresh questions, generator, context budget, and source-diversity policy. Retrieval recall and evidence recall can disagree because not every retrieved window fits in the prompt.

In [ ]:
for name, frame in frames['controlled'].items():
    display(frame.loc[[m for m in frame.index if m.startswith('retrieval_')], ['f1','retrieval_recall3','evidence_recall','coverage','p50_ms']])

## Retrieval-to-answer transfer

Answer quality need not rise with source recall. Wrong source windows, missing bridge evidence, generated claims unrelated to the question, and generation errors can break that connection.

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(11,4))
for ax,(name,frame) in zip(axes, frames['controlled'].items()):
    subset=frame.loc[[m for m in frame.index if m.startswith('retrieval_')]]
    ax.scatter(subset.evidence_recall, subset.f1)
    for label,row in subset.iterrows(): ax.annotate(label.replace('retrieval_',''),(row.evidence_recall,row.f1),fontsize=8)
    ax.set_title(name); ax.set_xlabel('Evidence recall'); ax.set_ylabel('Answer F1')
plt.tight_layout(); plt.show()

## Supporting-sentence evidence

Source-document recall can be high even when a selected window misses the annotated supporting sentence. This separate HotpotQA diagnostic requires a complete annotated sentence within a selected or accepted cited window. It does not reconstruct split sentences.

In [ ]:
sentences=read(OUT / 'sentence-evidence.json')
print(sentences['definition'])
display(pd.DataFrame(sentences['results']).T)

## Annotated answer-span availability

SQuAD source-document recall is nearly trivial because every question supplies one paragraph. Original answer-character spans provide a more useful check of whether selected and cited windows actually contain the annotated answer. Span availability does not prove support for a generated claim.

In [ ]:
spans=read(OUT / 'squad-answer-evidence.json')
print(spans['definition'])
display(pd.DataFrame(spans['results']).T)

## Context selection candidates

Development compares 384, 768, and 1,280 context tokens, repeated-source windows, and reversed evidence order. Full-prompt token counting enforces each model adapter's input limit.

In [ ]:
display(pd.DataFrame([{'label':r['label'],'f1':r['f1'],'evidence_recall':r['evidence_recall'],'p50_ms':r['p50_ms']} for r in selection['contexts']]))

## Context ablations

The controlled comparison separates context size, source diversity, and order. Source-diverse ordering visits a new document before taking another window from the same document.

In [ ]:
for name, frame in frames['controlled'].items():
    display(frame.loc[[m for m in frame.index if m.startswith('context_')], ['f1','evidence_recall','input_tokens','output_tokens','p50_ms']])

## Context sensitivity plot

More context can include useful evidence or distract the generator. Reversing order keeps the selected windows but changes their positions.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
for ax,(name,frame) in zip(axes,frames['controlled'].items()):
    frame.loc[[m for m in frame.index if m.startswith('context_')], 'f1'].plot.bar(ax=ax); ax.set_title(name); ax.set_ylim(0,1)
plt.tight_layout(); plt.show()

## Frozen combined configuration

The model, retrieval setting, context policy, and support thresholds are fixed before fresh evaluation. Balanced and strongest refer to resource tiers; neither label implies the highest held-out accuracy.

In [ ]:
display(selection['selected_configuration'])
print('Balanced: Qwen3-0.6B, BGE-small hybrid, 768-token context')
print('Strongest: Qwen3-1.7B with the development-selected retrieval and context policy')

## Expanded held-out results

The primary comparison contains 256 fresh questions per dataset, four times the original count. Raw structured answers, NLI-gated answers, and lexical gating are listed separately.

In [ ]:
for name,frame in frames['fresh'].items():
    display(frame[['n','em','f1','coverage','answerable_f1','unanswerable_recall','p50_ms']])

## Answer quality and coverage

A configuration that accepts fewer answers can change both overall F1 and the risk among accepted answers. The two quantities should remain visible together.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
for ax,(name,frame) in zip(axes,frames['fresh'].items()):
    frame[['f1','coverage']].plot.bar(ax=ax); ax.set_title(name); ax.set_ylim(0,1)
plt.tight_layout(); plt.show()

## Matched original before and after

The original 64 questions provide a direct comparison with the published baseline. They were excluded from new configuration selection. Fresh-sample results remain separate.

In [ ]:
before_after = read(OUT / 'before-after.json')
display(pd.DataFrame({name: {'original_hybrid_f1':r['original_hybrid']['f1'],'original_rerank_f1':r['original_rerank']['f1'],'upgraded_raw_f1':r['upgraded_raw']['f1'],'upgraded_verified_f1':r['upgraded_verified']['f1']} for name,r in before_after.items()}).T)

## Fresh baseline comparisons

The unchanged original baseline also runs on all 256 fresh questions. These paired comparisons avoid attributing a sample change to a model change. GPU and CPU latencies still have different backend conditions.

In [ ]:
for name in datasets:
    display(pd.DataFrame(baseline[name]['fresh_metrics']).T[['n','em','f1','context_recall','p50_ms']])
display(read(OUT / 'fresh-baseline-intervals.json'))

## Practical sequence-to-sequence reference

The plain-answer development winner is also evaluated through the preserved retrieval and post hoc citation pipeline on all fresh questions. Its lexical citations remain a known limitation. This comparison keeps answer quality separate from the structured-claim workflow.

In [ ]:
practical=read(OUT / 'practical-baseline.json')
print(practical['generator'],practical['selection'])
for name, splits in practical['datasets'].items():
    print(name)
    display(pd.DataFrame(splits['fresh']).T[['n','em','f1','citation_precision','citation_recall','p50_ms']])
display(practical['resources'])
display(read(OUT / 'practical-intervals.json'))
print(practical['device'],practical['generator_dtype'],practical['timing_scope'])

## Development calibration groups

Calibration fitting and threshold selection use disjoint development groups. SQuAD articles remain grouped. The logistic mapping predicts raw answer correctness and does not turn NLI scores into proof of entailment.

In [ ]:
for name, values in selection['thresholds'].items():
    assert not set(values['fit_ids']) & set(values['threshold_ids'])
    print(name, 'fit',len(values['fit_ids']),'threshold selection',len(values['threshold_ids']))
    display(values['calibrator'])

## Support threshold selection

Development answer F1 selects a threshold from a fixed grid. Lexical overlap uses the same development selection process on the same generated answers, allowing a controlled comparison with NLI support.

In [ ]:
for name, values in selection['thresholds'].items():
    print(name,'NLI threshold',values['selected'],'lexical threshold',values['lexical_selected'])
    display(pd.DataFrame(values['trials'])[['threshold','f1','coverage','answerable_f1','unanswerable_recall']])

## Abstention performance

Report answerable quality, unanswerable recall, abstention precision, coverage, and selective risk together. An always-empty system can score well on some mixtures while failing useful answer generation.

In [ ]:
for name,frame in frames['fresh'].items():
    display(frame.loc[['selected_raw','selected_verified','selected_lexical'], ['f1','answerable_f1','unanswerable_recall','abstention_precision','coverage','selective_risk']])

## Always-abstain control

Returning an empty answer to every question is a real baseline on the same held-out data. It reveals whether a high aggregate score mainly reflects the prevalence of unanswerable questions.

In [ ]:
display(pd.DataFrame(read(OUT / 'abstention-controls.json')).T)

## Held-out risk and coverage

These curves are descriptive after the threshold is frozen. Their highest point does not select the deployed cutoff.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4))
for ax,name in zip(axes,datasets):
    curve=pd.DataFrame(read(OUT / f'{name}-risk-coverage.json')['curve'])
    ax.plot(curve.coverage,curve.selective_risk,marker='o'); ax.set_title(name); ax.set_xlabel('Coverage'); ax.set_ylabel('Selective EM risk')
plt.tight_layout(); plt.show()

## Calibration scores

Brier score and ten-bin expected calibration error compare predicted raw correctness with actual exact match. They assess that target, not faithfulness or entailment.

In [ ]:
for name in datasets:
    calibration_report=read(OUT / f'{name}-calibration.json')
    display(pd.DataFrame({k:{x:v[x] for x in ('brier','ece10')} for k,v in calibration_report.items() if isinstance(v,dict)}).T)

## Calibration against a constant prior

A constant probability estimated from development can improve Brier score without using evidence. This comparator checks whether the fitted support mapping adds information beyond that prior.

In [ ]:
priors=read(OUT / 'calibration-prior-baseline.json')
display(pd.DataFrame({name:{'development_fit_n':r['development_fit_n'],'prior':r['smoothed_prior'],'brier':r['constant_prior']['brier'],'ece10':r['constant_prior']['ece10']} for name,r in priors.items()}).T)

## Support score and answer correctness

The histogram compares model support scores with exact answer correctness. A shared word, a valid ID, or a high NLI score cannot independently establish that an answer is correct.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4))
for ax,name in zip(axes,datasets):
    values=read(LOCAL / 'scores' / f'selected_raw-{name}-fresh.json')
    ax.hist([[v['support'] for v in values if v['raw_em']==1],[v['support'] for v in values if v['raw_em']==0]],bins=np.linspace(0,1,11),label=['Raw EM=1','Raw EM=0']); ax.set_title(name); ax.set_xlabel('NLI support'); ax.set_ylabel('Questions'); ax.legend()
plt.tight_layout(); plt.show()

## Reliability diagram

Sparse bins and shared article dependence limit calibration precision. The diagonal is a reference for perfect calibration on raw correctness.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(10,4))
for ax,name in zip(axes,datasets):
    bins=pd.DataFrame(read(OUT / f'{name}-calibration.json')['platt']['bins'])
    ax.scatter(bins.confidence,bins.accuracy,s=bins.n*3); ax.plot([0,1],[0,1],':'); ax.set_title(name); ax.set_xlabel('Predicted correctness'); ax.set_ylabel('Observed EM')
plt.tight_layout(); plt.show()

## Citation source agreement

Source precision and recall compare cited document IDs with annotations. They are distinct from NLI support and from the answer's correctness.

In [ ]:
for name,frame in frames['fresh'].items():
    display(frame[['citation_source_precision','citation_source_recall','citation_count','nli_claim_support_mean','nli_citation_support_mean']])

## Question-conditioned support follow-up

Matched original cases exposed a conservative answer-binding error and high NLI scores for facts about the wrong entity. A separate exploratory follow-up asks FLAN-T5-base to answer using only the cited windows, then checks agreement with the proposed answer. It reuses actual generations and adds real CPU verification; its latency is additional replay cost, not a new end-to-end timing pass.

In [ ]:
question_selection=read(OUT / 'answer-support-selection.json')
display(question_selection['selection'])
display(read(OUT / 'answer-support-adoption.json'))

## Question-conditioned held-out results

The follow-up's cutoffs use development groups only. It can recover supported yes/no claims rejected by the literal guard, but model agreement can share hallucinations and context limits can reject good answers. The original frozen protocol remains reported separately.

In [ ]:
question_results=read(OUT / 'answer-support-results.json')
for name, results in question_results['results'].items():
    display(pd.DataFrame({split:{k:v for k,v in values.items() if k!='paired_f1_vs_original_structured_parser'} for split,values in results.items()}).T)
    display(results['fresh']['paired_f1_vs_original_structured_parser'])
print(question_results['latency_scope'])

## Increment beyond the NLI rule

The incremental comparison uses the original development-selected NLI rule, rather than only the weaker raw parser. Cluster intervals crossing zero leave the direction uncertain. This comparison separates a useful point estimate from a reliable general improvement.

In [ ]:
question_analysis=read(OUT / 'answer-support-analysis.json')
comparison=pd.DataFrame({name:r['fresh']['question_check_vs_nli_f1'] for name,r in question_analysis.items()}).T
display(comparison)
fig,ax=plt.subplots(figsize=(8,3))
positions=np.arange(len(comparison))
ax.errorbar(comparison.delta,positions,xerr=np.vstack([comparison.delta-comparison.low,comparison.high-comparison.delta]),fmt='o'); ax.axvline(0,color='gray',linestyle=':'); ax.set_yticks(positions,comparison.index); ax.set_xlabel('Paired F1 change from NLI gating'); plt.tight_layout(); plt.show()
display(read(OUT / 'answer-support-resources.json'))

## Multiple source attribution

The engine permits multiple sources per claim and multiple claims. The actual frequency below shows whether the generator used that capacity. Support for only one side of a comparison is incomplete.

In [ ]:
for name in datasets:
    _,rs=records('selected_raw',name,'fresh')
    display(pd.Series([len({p['evidence'][s]['document_id'] for s in p['citations']}) for p in rs]).value_counts().sort_index().rename(name+' source count'))

## Parsing and support failures

Malformed output and fabricated citations are preserved locally and rejected by the acceptance path. A valid JSON object does not imply an accurate or supported answer.

In [ ]:
for name in datasets:
    _,rs=records('selected_raw',name,'fresh')
    display(pd.Series([p['reason'] for p in rs]).value_counts().rename(name+' parse outcomes'))
    print('Truncated answer calls:',sum(c['truncated'] for p in rs for c in p['calls'] if c['kind']=='answer'))

## Why the system abstains

A format failure can count as a correct empty answer on an unanswerable item without showing evidence understanding. Separate formatting and answer-binding rejection, model abstention, and NLI filtering by original answerability.

In [ ]:
for name,(_,test,_) in datasets.items():
    _,rs=records('selected_raw',name,'fresh')
    rows=[]
    for row,p in zip(test[64:],rs):
        accepted=gate(p,selection['thresholds'][name]['selected'])
        rows.append({'answerable':bool(row['answers']),'parse_reason':p['reason'],'final_flag':accepted['flag']})
    outcomes=pd.DataFrame(rows)
    display(pd.crosstab(outcomes.answerable,outcomes.parse_reason))
    display(pd.crosstab(outcomes.answerable,outcomes.final_flag))

## Evidence-dependent retrieval

The planner sees retrieved windows and generates follow-up questions about missing bridge or comparison facts. Each follow-up runs a new search. Invalid planner output produces no invented query and no frequent-word fallback.

In [ ]:
for name in datasets:
    _,rs=records('multistep',name,'controlled')
    print(name,'questions with actual follow-up retrieval',sum(len(p['retrieval_steps'])>1 for p in rs),'of',len(rs))
    example=next((p for p in rs if len(p['retrieval_steps'])>1),rs[0])
    display(example['retrieval_steps'])

## Rewrite, multi-step, and corrective comparisons

Rewrite uses the question alone. Multi-step retrieval uses first-pass evidence. Corrective retrieval adds a retry only after an insufficient initial result, accounting for both attempts. That retry also changes the deterministic generation seed. Without a generation-only retry control, its effect cannot be attributed entirely to new evidence.

In [ ]:
for name,frame in frames['controlled'].items():
    display(frame.loc[['rewrite','multistep','corrective'], ['f1','evidence_recall','retrieval_steps','retried','p50_ms','planning_ms','output_tokens']])

## Corrective retry traces

A retry can recover useful evidence or amplify an early mistake. The records show whether a retry ran, its follow-up queries, and the final support score. Retrieval never accesses gold evidence. Initial answer text is retained, but its individual NLI scores are not stored separately.

In [ ]:
for name in datasets:
    _,rs=records('corrective',name,'controlled')
    print(name,'retries',sum(p['retried'] for p in rs),'of',len(rs))
    example=next((p for p in rs if p['retried']),rs[0])
    display({'retrieval_steps':example['retrieval_steps'],'support':example['support'],'flag':example['flag'],'timings':example['timings']})

## Paired cluster uncertainty

Paired percentile intervals resample full SQuAD articles and HotpotQA questions. They are exploratory with no correction for multiple comparisons. Questions sharing background knowledge can remain dependent. The first intervals use the selected raw configuration as reference. Family comparisons below use plain FLAN-T5-small, BM25, or the 768-token diverse context as named references. Generator tokenizers can still change the packed evidence.

In [ ]:
intervals=read(OUT / 'paired-intervals.json')
for name,values in intervals.items(): display(pd.DataFrame(values).T)
family=read(OUT / 'controlled-family-intervals.json')
for name,comparisons in family['datasets'].items():
    print(name)
    display(comparisons)
for name, methods in intervals.items():
    reference=read(LOCAL / 'scores' / f'selected_raw-{name}-fresh.json')[:config['ablation_questions_per_dataset']]
    for label, expected in methods.items():
        values=read(LOCAL / 'scores' / f'{label}-{name}-controlled.json')
        actual=cluster_interval([v['f1'] for v in reference],[v['f1'] for v in values],[v['group'] for v in values])
        assert actual==expected
print('Every controlled paired interval reconciles with retained scores')

## Independent metric reconciliation

Every fresh selected aggregate is recalculated from original prediction artifacts. This checks answer scoring, accepted citations, source recall, and all recorded stage times.

In [ ]:
for name,(_,test,_) in datasets.items():
    _,rs=records('selected_raw',name,'fresh')
    raw=[per_question(row,p) for row,p in zip(test[64:],rs)]
    verified=[per_question(row,p,selection['thresholds'][name]['selected']) for row,p in zip(test[64:],rs)]
    assert aggregate(raw)==metrics['fresh'][name]['selected_raw']
    assert aggregate(verified)==metrics['fresh'][name]['selected_verified']
print('Fresh selected metrics reconcile exactly')

## Stage latency

Query encoding, retrieval, reranking, context assembly, planning, generation, and verification have separate timers. GPU stages synchronize before elapsed times are recorded. Indexing and loading remain outside per-answer totals.

In [ ]:
columns=['query_encoding_ms','retrieval_ms','reranking_ms','context_ms','planning_ms','generation_ms','verification_ms']
for name,frame in frames['fresh'].items(): display(frame[columns+['p50_ms','p95_ms']])

## Per-question latency spread

Latency varies with context, output length, and model behavior. These distributions show the measured sequential pass rather than repeated-run performance estimates.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4))
for ax,name in zip(axes,datasets):
    labels=['selected_raw','balanced','strongest']
    values=[read(LOCAL / 'scores' / f'{label}-{name}-fresh.json') for label in labels]
    ax.boxplot([[v['total_ms'] for v in rows] for rows in values],tick_labels=labels); ax.set_title(name); ax.set_ylabel('Milliseconds')
plt.tight_layout(); plt.show()

## Uncached repeatability probe

Two fresh inference passes on eight development questions check fixed-seed output agreement and expose timing variation. They are separate from the held-out benchmark and do not use cached answers.

In [ ]:
display(read(OUT / 'repeatability.json'))

## Synthetic evidence-handling failures

These five cases are engineering probes, separate from benchmark totals. Only two return the expected answer. Invalid JSON rejects the clean, injected-instruction, and two-source cases. Rejecting both clean and injected text does not establish useful injection resistance.

In [ ]:
display(pd.DataFrame(read(OUT / 'security-probes.json')['results']))

## CLI integration checks

Two real CLI calls use a synthetic negation example. The second sets an NLI cutoff above one to force abstention and check that the override reaches the selected verification path. This small check does not establish general answer quality.

In [ ]:
display(read(OUT / 'cli-smoke.json'))

## Latency composition plot

Stage averages expose whether a gain comes with expensive generation, verification, or repeated retrieval. Total latency also contains host-side work between stages.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,4))
for ax,(name,frame) in zip(axes,frames['fresh'].items()):
    frame.loc[['selected_raw','balanced','strongest'],columns].plot.bar(stacked=True,ax=ax); ax.set_title(name); ax.set_ylabel('Mean milliseconds')
plt.tight_layout(); plt.show()

## Token counts and throughput

Output-token counts include planning and every answer attempt. Throughput is computed from recorded inference times, not cache reads. Local inference has no API charge.

In [ ]:
for name,frame in frames['fresh'].items(): display(frame[['input_tokens','output_tokens','questions_per_second','generated_tokens_per_second']])

## Peak resources and loading

The observer records sampled process RSS, Windows process peak memory, and GPU allocation/reservation peaks. These are whole-process observations, not isolated method peaks. NLI truncation counts describe the executed inference calls.

In [ ]:
resources=read(OUT / 'resources.json')
display(resources)
host=read(OUT / 'host-load.json')
display({k:v for k,v in host.items() if k!='gpu_snapshot'})
print(host['gpu_snapshot'])

## Index construction

Document encoding and lexical index construction are recorded separately for each question's candidate pool. Reusing those indexes within a run avoids confusing repeated construction with answer latency.

In [ ]:
for name in datasets:
    _,rs=records('selected_raw',name,'fresh')
    display(pd.DataFrame([p['indexing'] for p in rs]).describe())

## Matched support case review

The new assistant review uses actual cited windows on matched original questions and distinguishes supported, partial, unsupported, and abstained outcomes. It is neither blinded nor independent human annotation.

In [ ]:
review=read(OUT / 'support-review.json')
display({k:v for k,v in review.items() if k!='cases'})
display(pd.DataFrame(review['cases']))

## Representative failures

The first three zero-F1 selected responses in deterministic fresh order are shown locally. The public notebook strips these outputs, retaining no raw benchmark text.

In [ ]:
for name,(_,test,_) in datasets.items():
    _,rs=records('selected_raw',name,'fresh')
    failures=[]
    for row,p in zip(test[64:],rs):
        final=gate(p,selection['thresholds'][name]['selected'])
        if answer_scores(final['final_answer'],row['answers'])[1]==0:
            failures.append({'id':row['id'],'question':row['question'],'references':row['answers'],'answer':final['final_answer'],'raw_answer':p['answer'],'support':p['support'],'reason':final['flag']})
    display(pd.DataFrame(failures[:3]))

## Question-type failures

Bridge, comparison, answerable, and unanswerable strata can reveal different failure modes. No stratum-specific configuration was tuned on held-out results.

In [ ]:
for name in datasets:
    scores=read(LOCAL / 'scores' / f'selected_verified-{name}-fresh.json')
    display(pd.DataFrame(scores).groupby('kind')[['em','f1','answered','support']].agg(['count','mean']))

## Unsuccessful experiments

Higher-capacity models and additional retrieval stages can perform worse. These differences use the same controlled question subset. Paired intervals and coverage should accompany the point estimates.

In [ ]:
for name,frame in frames['controlled'].items():
    matched=read(LOCAL / 'scores' / f'selected_raw-{name}-fresh.json')[:config['ablation_questions_per_dataset']]
    reference=np.mean([r['f1'] for r in matched])
    display(frame[['f1','coverage','p50_ms']].assign(difference_from_matched_selected=frame.f1-reference))
print('All differences use matched question subsets; paired cluster intervals are reported above.')

## Publication and remaining limits

The executed upgrade notebook remains local. Publication clears a deep copy after verifying complete execution and identical cell sources. The original executed notebook is never overwritten. Model-based support can miss question relevance, truncation can remove evidence, and valid citations can accompany wrong answers. Larger or independent annotation studies are still needed.

In [ ]:
preserved()
print('Original baseline bytes preserved')
print('Selected model:',selection['selected_generator'])
for name,frame in frames['fresh'].items():
    print(name,'raw F1',round(frame.loc['selected_raw','f1'],4),'verified F1',round(frame.loc['selected_verified','f1'],4),'coverage',round(frame.loc['selected_verified','coverage'],4))